# Visual anomaly detection
This lab is based on the paper by Vitjan Zavrtanik, Matej Kristan and Danijel Skočaj: [*Reconstruction by inpainting for visual anomaly detection*](https://doi.org/10.1016/j.patcog.2020.107706), Pattern Recognition, 2021.


In [ ]:
! wget https://www.mydrive.ch/shares/150996/b52ecdcbf521176e9db9c731f2304b27/download/420938113-1629960298/mvtec_anomaly_detection.tar.xz
! rm -rf sample_data/
!tar -xf mvtec_anomaly_detection.tar.xz
!pip install -q lightning torchmetrics
!git clone https://github.com/NyxAether/RIAD_models.git

In [ ]:
! nvidia-smi
! find */train/ -name *.png | wc -l

In [ ]:
from pathlib import Path

import cv2
import lightning
import matplotlib.pyplot as plt
import numpy as np
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics.functional.image import (
  multiscale_structural_similarity_index_measure as ms_ssim,
)
from tqdm.notebook import tqdm, trange

## Useful Functions

In [ ]:
RESHAPE_DIM = (256, 256)
device = "cuda" if torch.cuda.is_available() else "cpu"


def load_data(files):
  nb_files = len(files) - len(files) % 10
  data = np.zeros((nb_files, *RESHAPE_DIM, 3), dtype="float32")
  for i in trange(nb_files):
    im = cv2.imread(str(files[i]))
    im = cv2.resize(im, RESHAPE_DIM)
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    data[i, :, :, :] = im
  data = data / 255
  # PyTorch expects the channels right after the batch dimension
  data_torch = torch.from_numpy(data).permute(0, 3, 1, 2).contiguous()
  return data_torch


def to_displayable(image: torch.Tensor) -> np.ndarray:
  """Turn a (channels, height, width) tensor into an array imshow can plot."""
  image = image.detach().cpu()
  return image.permute(1, 2, 0).numpy() if image.dim() == 3 else image.numpy()


@torch.no_grad()
def predict(model: nn.Module, data: torch.Tensor, batch_size: int = 5) -> torch.Tensor:
  """Apply a model to a batch of images, batch by batch."""
  model.eval()
  return torch.cat([model(batch.to(device)).cpu() for batch in data.split(batch_size)])


def display(data, is_mask=True):
  shape = data.shape
  if is_mask:
    h, w = shape[0], shape[1]
    fig, ax = plt.subplots(h, w, dpi=150)
    for l in range(h):
      for c in range(w):
        ax[l, c].set_xticks([])
        ax[l, c].set_yticks([])
        ax[l, c].imshow(to_displayable(data[l, c]), cmap="gray")
  else:
    h, w = shape[0], 1
    fig, ax = plt.subplots(h, w, dpi=150)
    for i, axi in enumerate(ax):
      axi.set_xticks([])
      axi.set_yticks([])
      axi.imshow(to_displayable(data[i]), cmap="gray")

## List of Objects

In [ ]:
dossier_courant = Path()
liste_objets = [
  d.name for d in dossier_courant.iterdir() if d.name[0] != "." and not d.is_file()
]
liste_objets.remove("RIAD_models")
for obj in liste_objets:
  print(obj)

## Fetching the Data for an Object
 * The training data
 * The test data, including anomalous and valid cases
 * The anomaly detection masks

All the data will be converted into 256*256 images.

In [ ]:
objet = "transistor"


def load_all(objet):
  # Training data
  files_train = sorted(list(Path(f"{objet}/train/good/").glob("*.png")))
  nb_files = len(files_train) - len(files_train) % 10
  data = load_data(files_train)

  # Defective data
  defect_path = set(Path(f"{objet}/test/").glob("*")) - set(
    Path(f"{objet}/test/").glob("good")
  )
  files_defect = set()
  for p in defect_path:
    files_defect = files_defect | set(p.glob("*.png"))
  files_defect = sorted(files_defect)
  defect_data = load_data(files_defect)

  # Valid data
  files_good = sorted(list(Path(f"{objet}/test/good/").glob("*.png")))
  goods = load_data(files_good)

  # Ground truth masks for the defective data
  files_ground = sorted(list(Path(f"{objet}/ground_truth/").glob("*/*.png")))
  ground = load_data(files_ground)
  return data, defect_data, goods, ground


data, defect_data, goods, ground = load_all(objet)

## Masking the Images
The anomaly detection technique relies on generating masked image fragments.

To do so, we generate $n$ masks on a given image, following two rules:
 * The image is randomly masked by blocks of $k\times k$ pixels (usually a power of two, but in theory any divisor of the image size)
 * The masks don't overlap and hide the entire image when stacked (1 pixel is masked by at most one mask, but at least one)

In [ ]:
def generate_masks(images_shape: tuple, k: int = 2, n: int = 3) -> torch.Tensor:
  """images_shape: shape of a batch of images
  k: size of hidding block
  n: number of masks
  """
  shape = (images_shape[0], *(ax // k for ax in images_shape[-2:]))
  indices = torch.randint(0, n, shape)
  masks = 1 - nn.functional.one_hot(indices, n).float()
  masks = masks.permute(0, 3, 1, 2)
  return nn.functional.interpolate(masks, images_shape[-2:], mode="nearest")


def apply_masks(img: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
  # (b, 1, c, h, w) * (b, m, 1, h, w) -> (b, m, c, h, w)
  return img.unsqueeze(-4) * masks.unsqueeze(-3)


def reconstruct_image(imgs: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
  invert_masks = (1 - masks).unsqueeze(-3)
  return torch.sum(imgs * invert_masks, dim=-4)

`generate_masks` produces random masks for each image, and `apply_masks` duplicates an image and applies the created masks to each of the copies.
To see the result, let's apply these methods to two images.

**You can change the size of the masks $k$ and the number of masks $n$**

In [ ]:
# Generate masks
masks = generate_masks(data[:2].shape, k=16, n=3)
# Apply the masks to the images
data_masked = apply_masks(data[:2], masks)

display(data[:2], is_mask=False)

In [ ]:
display(masks)

In [ ]:
display(data_masked)

The masks are exclusive, meaning they never cover the same part of the image. The model's goal will be to fill in the missing parts of the masked data, then reassemble them.

In [ ]:
inverse_masked = apply_masks(data[:2], 1 - masks)
display(inverse_masked)

## Loss Function

The method's loss function is computed from the *Gradient magnitude similarity* and the *Structural similarity index*.

### Gradient magnitude similarity (GMS)

The GMS relies on computing a gradient $g(I)$ on the images considered:

$$g(I)=\sqrt{(I*h_x)^2+(I*h_y)^2}$$

with $*$ the convolution operator, $I$ the image, and the pair $h_x$ and $h_y$ the Prewitt filters along the $x$ and $y$ dimensions, usually:

$h_x=\begin{bmatrix}
-1 & 0 & +1 \\
-1 & 0 & +1 \\
-1 & 0 & +1
\end{bmatrix}$ and $h_y=\begin{bmatrix}
-1 & -1 & -1 \\
0 & 0 & 0 \\
+1 & +1 & +1
\end{bmatrix}$

$g(I)$ is therefore the Euclidean norm of a horizontal and a vertical gradient.

In [ ]:
# Vectorized
def prewitt_filter(img: torch.Tensor, c=1e-9) -> torch.Tensor:
  # Define kernel for x differences
  if img.dim() < 4:
    im = img.reshape((1,) + img.shape)
  else:
    im = img
  kx = (
    torch.tensor([[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]])
    .to(im)
    .repeat(3, 1, 1, 1)
  )
  # Define kernel for y differences
  ky = (
    torch.tensor([[-1.0, -1.0, -1.0], [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]])
    .to(im)
    .repeat(3, 1, 1, 1)
  )
  # Perform x convolution
  img_prewittx = nn.functional.conv2d(im, kx, padding="same", groups=3) / 3
  # Perform y convolution
  img_prewitty = nn.functional.conv2d(im, ky, padding="same", groups=3) / 3

  gmm = torch.sqrt(img_prewittx * img_prewittx + img_prewitty * img_prewitty + c)

  if img.dim() < 4:
    gmm = gmm.reshape(img.shape)
  return gmm

In [ ]:
filtered_im = prewitt_filter(data[0])
display(torch.stack([data[0], filtered_im]), is_mask=False)

Using $g(I)$, we'll compute the GMS similarity metric defined between two images $I$ and $I_r$, respectively the original image and the image reconstructed by the autoencoder:

$GMS(I,I_r)= \frac{2g(I)g(I_r)+c}{g(I)^2+g(I_r)^2+c}$

with $c$ a constant ensuring numerical stability.

In [ ]:
# Vectorized.
def GMS(I: torch.Tensor, Ir: torch.Tensor, c=1e-9) -> torch.Tensor:
  gI = prewitt_filter(I)
  gIr = prewitt_filter(Ir)
  gms = (2 * gI * gIr + c) / (torch.square(gI) + torch.square(gIr) + c)
  return gms


# Tensor of 1s if identical
print(torch.mean(GMS(data[:2], data[:2]), dim=(-1, -2, -3)))
# Tensor different from 1 if different
print(torch.mean(GMS(data[:2], data[2:4]), dim=(-1, -2, -3)))

### The $L_G$ Loss Function
Since the GMS is a tensor, we compute the loss function $L_G$ as the sum of the complements of the GMS:

$$L_G(I,I_r)=\sum^H_{i=1}\sum^W_{i=1} 1 - GMS(I,I_r)$$

with $H$ and $W$ the height and width of the image.

In [ ]:
# Vectorized. Take one image per argument
def L_G(I: torch.Tensor, Ir: torch.Tensor) -> torch.Tensor:
  if I.dim() < 4:
    return torch.mean(1 - GMS(I, Ir))
  return torch.mean(1 - GMS(I, Ir), dim=(-1, -2, -3))


# Zero when identical
print(L_G(data[:2], data[:2]))
# Non-zero when different
# but <dim*dim2
print(L_G(data[:2], data[2:4]))

Rather than using GMS, we'll use its multiscale version $MSGMS(I,I_r)$, that is, deliberately reduce the size of the compared images several times and apply GMS each time. The results are then averaged. This will in particular "*smooth*" the results.

In the original paper, MSGMS is computed with an average pooling of (2), down to an eighth of the initial dimension.

$$ MSMGS(I,Ir) = \frac{1}{|S|}\sum_{s\in S} GMS(I_s,Ir_{s})$$

with $S$ the set of reduced image formats, $I_s$ and $Ir_s$ the images $I$ and $Ir$ resized to format $s$

In [ ]:
# Vectorized.
def MSGMS(I: torch.Tensor, Ir: torch.Tensor, is_loss=True) -> torch.Tensor:
  total_loss = GMS(I, Ir)

  for _ in range(3):
    I = nn.functional.avg_pool2d(I, 2, stride=2)
    Ir = nn.functional.avg_pool2d(Ir, 2, stride=2)
    total_loss = total_loss + nn.functional.interpolate(
      GMS(I, Ir), total_loss.shape[-2:], mode="bilinear"
    )

  total_loss = total_loss / 4
  if is_loss:
    return 1 - torch.mean(total_loss, dim=(-1, -2, -3))
  return total_loss


# Vectorized version
print(MSGMS(data[:2], data[:2]))
print(MSGMS(data[:2], data[2:4]))

### Structural Similarity Index (SSIM)
The SSIM index is a similarity measure dedicated to images, defined by [Wang et al.](https://www.cns.nyu.edu/pub/eero/wang03-reprint.pdf).

It takes 3 different criteria into account to establish similarity:
 * luminance
 * contrast
 * overall structure

Fortunately, the method is already implemented in the [**torchmetrics.functional.image.multiscale_structural_similarity_index_measure**](https://lightning.ai/docs/torchmetrics/stable/image/multi_scale_structural_similarity.html) package, imported here as `ms_ssim`. We'll therefore use the SSIM like the GMS and compute a loss function $L_S$ that is the sum of the complements of the similarities:

$L_S(I,I_r)=\sum^H_{i=1}\sum^W_{i=1} 1 - SSIM(I,I_r)$

In [ ]:
# Vectorized.
def L_S(I: torch.Tensor, Ir: torch.Tensor, max_val=1.0) -> torch.Tensor:
  # Max val is the maximum difference between values allowed
  matrix_ssim = ms_ssim(I, Ir, data_range=max_val, reduction="none")
  return 1 - matrix_ssim


# Zero when identical
print(L_S(data[:2], data[:2]))
# Non-zero when different
print(L_S(data[:2], data[2:4]))

### Final Loss Function
The loss function $L$ we'll use will simply be the weighted sum of the loss functions $L_G$ and $L_S$:

$L=λ_G L_G + λ_S L_S + L2$

with $λ_G,λ_S$ the weights of the respective loss functions $L_G, L_S$, and $L2$ the regularization of the autoencoder.

In [ ]:
def loss_vad(
  y: torch.Tensor, y_hat: torch.Tensor, lambda_G: float = 1, lambda_S: float = 1
) -> torch.Tensor:
  return torch.sum(
    lambda_G * MSGMS(y, y_hat)
    + lambda_S * L_S(y, y_hat)
    + torch.mean((y - y_hat) ** 2, dim=(-1, -2, -3))
  )


# Zero when identical
print(loss_vad(data[:2], data[:2]))
# Non-zero when different
print(loss_vad(data[:2], data[2:4]))

## Creating the Model
The RIAD model thus applies the masking technique to find the anomalies that may be present in an image.

During training, we apply the following algorithm:
 * Randomly choose a block size $k$ for the masks (among the values 2, 4, 8 and 16)
 * Randomly generate the masks and apply them to the images
 * For each masked image, the autoencoder tries to reproduce the initial image.
 * The masked parts generated by the autoencoder are reassembled into a single image, which is then compared to the original image

The rest is standard

In [ ]:
class RIAD(lightning.LightningModule):
  def __init__(self, model: nn.Module, learning_rate: float = 1e-3) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model

  def forward(self, x):
    return self.model(x)

  def training_step(self, batch, batch_index):
    (x,) = batch
    # Block size 2**k among 2, 4, 8 and 16, as at inference
    k = torch.randint(1, 5, (1, 1))[0, 0]
    masks = generate_masks(x.shape, 2**k, 3).to(x)
    nb_masks = masks.shape[1]
    masked = apply_masks(x, masks)

    partial_xs = []
    for i in range(nb_masks):
      partial_xs.append(self(masked[:, i]))  # Forward pass

    x_r = reconstruct_image(torch.stack(partial_xs, dim=1), masks)
    loss = loss_vad(x, x_r)
    loss_g = MSGMS(x, x_r).mean()
    loss_s = L_S(x, x_r).mean()
    l2 = torch.mean((x - x_r) ** 2)

    # Log a dict mapping metric names to current value
    self.log_dict(
      {"loss": loss, "lossg": loss_g, "loss_s": loss_s, "l2": l2},
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    return loss

  def configure_optimizers(self):
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

  @torch.no_grad()
  def map_error(self, data):
    """Generate the heatmap from the MSGMS score
    """
    mask_sizes = (2, 4, 8, 16)
    self.eval()
    data = data.to(self.device)
    Ga = torch.zeros_like(data)
    avg_filter = torch.ones(1, 3, 21, 21).to(data) / (21 * 21 * 3)
    for k in mask_sizes:
      masks = generate_masks(data.shape, k, 3).to(data)
      nb_masks = masks.shape[1]
      masked = apply_masks(data, masks)

      partial_xs = []
      for i in range(nb_masks):
        partial_xs.append(self(masked[:, i]))  # Forward pass

      x_r = reconstruct_image(torch.stack(partial_xs, dim=1), masks)
      Ga = Ga + 1 - MSGMS(data, x_r, False)
    Ga = Ga / len(mask_sizes)
    Ga = nn.functional.conv2d(Ga, avg_filter, padding="same")
    return Ga

### UNet Network

In [ ]:
def downsample() -> nn.MaxPool2d:
  return nn.MaxPool2d(kernel_size=(2, 2))


def upsample() -> nn.Upsample:
  return nn.Upsample(scale_factor=2)


def semiblock(in_channels, feature_maps):
  conv = nn.Conv2d(in_channels, feature_maps, 3, padding="same")
  batch_norm = nn.BatchNorm2d(feature_maps)
  return nn.Sequential(conv, nn.ReLU(), batch_norm, nn.ReLU(), nn.Dropout(0.1))


def block(in_channels, feature_maps):
  return nn.Sequential(
    semiblock(in_channels, feature_maps), semiblock(feature_maps, feature_maps)
  )


class UNet(nn.Module):
  """U-net: an encoder, a decoder, and skip connections between the two."""

  def __init__(
    self, in_channels: int = 3, starting_depth: int = 64, steps: int = 5
  ) -> None:
    super().__init__()
    depths = [min(512, starting_depth * 2**i) for i in range(steps)]

    self.downsample = downsample()
    self.upsample = upsample()

    self.encoder = nn.ModuleList(
      [
        block(depths[i - 1] if i else in_channels, depth)
        for i, depth in enumerate(depths)
      ]
    )

    # At each decoder level, the encoding of the same resolution is concatenated
    # to the upsampled tensor, hence the number of input channels
    self.decoder = nn.ModuleList(
      [
        block(depth + previous_depth, depth)
        for depth, previous_depth in zip(depths[-2::-1], depths[::-1])
      ]
    )

    self.outputs = nn.Conv2d(depths[0], 3, 1)

  def forward(self, inputs: torch.Tensor) -> torch.Tensor:
    encodings = []
    for i, encoder_block in enumerate(self.encoder):
      encodings.append(encoder_block(self.downsample(encodings[-1]) if i else inputs))

    x = encodings[-1]
    for decoder_block, encoding in zip(self.decoder, encodings[-2::-1]):
      x = decoder_block(torch.cat([encoding, self.upsample(x)], dim=1))

    return torch.sigmoid(self.outputs(x))


def unet(pretrained_weights=None, input_size=(3, 256, 256), starting_depth=64, steps=5):
  model = RIAD(UNet(input_size[0], starting_depth, steps))

  # print(ModelSummary(model, max_depth=-1))

  if pretrained_weights:
    model.load_state_dict(torch.load(pretrained_weights))

  return model


# Build model
model = unet()

## Training
__WARNING__: training can take a very long time.

You can go straight down to the `Visualization with pre-trained models` part if you just want to see the result.

In [ ]:
callbacks = [
  EarlyStopping(
    monitor="loss",
    patience=20,
    check_on_train_epoch_end=True,
  ),
  ModelCheckpoint(monitor="loss", mode="min", save_top_k=1, filename="best_model"),
]
train_loader = DataLoader(TensorDataset(data), batch_size=5, shuffle=True)
trainer = lightning.Trainer(
  max_epochs=1000,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="riad"),
  callbacks=callbacks,
)
trainer.fit(model, train_loader)

# Reload the weights of the best model
model.load_state_dict(
  torch.load(trainer.checkpoint_callback.best_model_path)["state_dict"]
)

## Evaluations
Let's look at the results of our training

In [ ]:
def error_vis(im, pred, err_map, ground=None, threshold=0.5, name=None):
  fig, ax = plt.subplots(
    len(im), 3 if ground is None else 4, figsize=(9, 4 * len(im)), squeeze=False
  )
  for i in range(len(im)):
    for axi in ax[i]:
      axi.set_xticklabels([])
      axi.set_yticklabels([])
    ax[i, 0].imshow(to_displayable(im[i]))
    ax[i, 0].imshow(to_displayable(err_map[i][0, 0]), alpha=0.5, cmap="jet")

    ax[i, 1].imshow(to_displayable(pred[i]))
    ax[i, 2].imshow(to_displayable(im[i]))

    if ground is not None:
      ax[i, 3].imshow(to_displayable(im[i]))
      ax[i, 3].imshow(to_displayable(ground[i]), cmap="jet", alpha=0.3)
  fig.tight_layout()
  if name != None:
    fig.savefig(f"{name}.png")
  else:
    fig.show()


def display_err(objet: str, max_object: int = 5, good_data=False):
  model = unet(f"RIAD_models/best_model_{objet}.pt")
  model.to(device)
  _, defect_data, good, ground = load_all(objet)
  defect_data = defect_data[:max_object]
  ground = ground[:max_object]
  good = good[:max_object]
  errors = []
  if good_data:
    for d in tqdm(good):
      errors.append(model.map_error(d[None, :]))

    res = predict(model, good)
    error_vis(good, res, errors, name=objet)
  else:
    for d in tqdm(defect_data):
      errors.append(model.map_error(d[None, :]))

    res = predict(model, defect_data)
    error_vis(defect_data, res, errors, ground, name=objet)

### Data With Anomalies

In [ ]:
errors = []
for d in tqdm(defect_data):
  errors.append(model.map_error(d[None, :]))

res = predict(model, defect_data)

In [ ]:
error_vis(defect_data, res, errors, ground)

### Data Without Anomalies

In [ ]:
err_goods = []
for d in tqdm(goods):
  err_goods.append(model.map_error(d[None, :]))

res_goods = predict(model, goods)

In [ ]:
error_vis(goods, res_goods, err_goods)

## Visualization With Pre-Trained Models

### Data With Anomalies

In [ ]:
objet = "carpet"
display_err(objet)

### Data Without Anomalies

In [ ]:
display_err(objet, good_data=True)